In [1]:
import os

In [80]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredRSTLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
class UnstructuredRSTLoaderWithMetadata(UnstructuredRSTLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as UnstructuredRSTLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using UnstructuredRSTLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [75]:
def metadata_func(document, metadata):
    # extract the page title from the content and store it in metadata
    title = document.page_content.split('\n')[0].split('===')[0]
    metadata['title'] = title
    return metadata

In [ ]:
loader = DirectoryLoader(
    path='../data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=UnstructuredRSTLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
        'mode': 'single',
    },
    show_progress=True,
    use_multithreading=True
)

In [93]:
docs = loader.load()

 35%|███▌      | 11/31 [00:01<00:02,  7.38it/s][WARNING] Could not convert TeX math \text{FPC} = \sqrt{(N-n)/(N-1)}, rendering as TeX
[WARNING] Could not convert TeX math \text{SE}_{\hat{p}} = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \text{center} = \frac{\hat{p} + z^2/(2n_{\text{eff}})}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \text{margin} = \frac{z\sqrt{\hat{p}(1-\hat{p})/n_{\text{eff}} + z^2/(4n_{\text{eff}}^2)}}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon = (b-a)\sqrt{\frac{\ln(2/\alpha)}{2n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon_{\bar{X}} = R\sqrt{\ln(2/\alpha)/(2n)}, rendering as TeX

100%|██████████| 31/31 [00:03<00:00,  8.58it/s]


In [98]:
# text splitter 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
     encoding_name='gpt2',
     chunk_size=512, 
     chunk_overlap=32
)